In [1]:
# ==============================
# 1. Install & Import Libraries
# ==============================
!pip install torch torchvision --quiet

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from PIL import Image, ImageFile
from collections import Counter
from tqdm import tqdm
import numpy as np

In [2]:
# ==============================
# 2. Mount Google Drive (new account)
# ==============================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
!ls "/content/drive/MyDrive/Resnet-18_Folder/ResNet_Checkpoints"


epoch1_batch100.pth  epoch1_batch200.pth  latest.pth


In [9]:
# ==============================
# 3. Paths
# ==============================
# Path to the folder that contains all checkpoints
checkpoint_dir = "/content/drive/MyDrive/Resnet-18_Folder/ResNet_Checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
latest_checkpoint = os.path.join(checkpoint_dir, "latest.pth")
log_file = os.path.join(checkpoint_dir, "training_log.txt")

# Dataset path
dataset_path = "/content/drive/MyDrive/Dataset/SMF-DataSet"

In [10]:
# ==============================
# 4. Data Transforms
# ==============================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Safe image loader
def safe_loader(path):
    try:
        img = Image.open(path).convert("RGB")
        return img
    except Exception as e:
        print(f"⚠️ Skipping corrupted image: {path}, error: {e}")
        return Image.new("RGB", (224, 224), (0, 0, 0))

In [11]:
# ==============================
# 5. Dataset
# ==============================
full_dataset = datasets.ImageFolder(dataset_path, transform=transform, loader=safe_loader)

# Check class distribution
counts = Counter([label for _, label in full_dataset.samples])
print("Dataset distribution per class:", counts)
print("Classes:", full_dataset.classes)

# Train/Validation split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# WeightedRandomSampler for class balance
train_indices = train_dataset.indices
train_labels = [full_dataset.samples[i][1] for i in train_indices]
label_counts = Counter(train_labels)
class_weights = {cls: 1.0 / count for cls, count in label_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]
sample_weights = torch.DoubleTensor(sample_weights)

train_sampler = WeightedRandomSampler(weights=sample_weights,
                                      num_samples=len(sample_weights),
                                      replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, sampler=train_sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

num_classes = len(full_dataset.classes)
print("Number of classes:", num_classes)


Dataset distribution per class: Counter({8: 16677, 0: 13651, 9: 12917, 7: 11911, 2: 9566, 5: 9408, 6: 5715, 3: 5585, 4: 2718, 1: 2651})
Classes: ['bbq', 'bread', 'desserts', 'drinks', 'lentil', 'meat', 'misc', 'rice', 'street', 'vegetable']
Number of classes: 10


In [12]:
# ==============================
# 6. Define Model (ResNet-18)
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


Using device: cuda


In [17]:
# 7. Resume Training if Checkpoint Exists
# ==============================
start_epoch = 0
start_batch = 0
num_epochs = 13
batch_save_interval = 100

if os.path.exists(latest_checkpoint):
    print("🔄 Loading latest checkpoint...")
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    start_epoch = checkpoint['epoch']
    start_batch = checkpoint.get('batch', -1) + 1  # start from next batch
    print(f"✅ Resuming from epoch {start_epoch+1}, batch {start_batch+1}")
else:
    print("⚠️ No checkpoint found, starting from scratch")
    start_epoch = 0
    start_batch = 0


🔄 Loading latest checkpoint...
✅ Resuming from epoch 1, batch 301


In [ ]:
# ==============================
# Full Training Loop with Proper Batch & Epoch Checkpointing
# ==============================
import os
from tqdm import tqdm
import torch

# ------------------------------
# Paths & Parameters
# ------------------------------
checkpoint_dir = "/content/drive/My Drive/Resnet-18_Folder/ResNet_Checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
latest_checkpoint = os.path.join(checkpoint_dir, "latest.pth")
log_file = os.path.join(checkpoint_dir, "training_log.txt")

num_epochs = 13
batch_save_interval = 100

# ------------------------------
# Resume from latest checkpoint
# ------------------------------
start_epoch = 0
start_batch = 0

if os.path.exists(latest_checkpoint):
    print("🔄 Loading latest checkpoint...")
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    start_epoch = checkpoint['epoch']
    start_batch = checkpoint.get('batch', 0) + 1  # start from next batch
    print(f"✅ Resuming from epoch {start_epoch+1}, batch {start_batch+1}")
else:
    print("⚠️ No checkpoint found, starting from scratch")

# ------------------------------
# Training loop
# ------------------------------
for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    loop = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (images, labels) in loop:
        # Skip batches already done
        if epoch == start_epoch and batch_idx < start_batch:
            continue

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update tqdm
        loop.set_postfix({
            "Batch Loss": f"{running_loss/(batch_idx+1):.4f}",
            "Acc": f"{100*correct/total:.2f}%"
        })

        # --- Save batch-level checkpoint every batch_save_interval ---
        if (batch_idx + 1) % batch_save_interval == 0:
            batch_checkpoint_path = os.path.join(checkpoint_dir, f"epoch{epoch+1}_batch{batch_idx+1}.pth")
            torch.save({
                'epoch': epoch,
                'batch': batch_idx,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
            }, batch_checkpoint_path)

            # Update latest checkpoint
            torch.save({
                'epoch': epoch,
                'batch': batch_idx,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
            }, latest_checkpoint)

    # --- Epoch metrics ---
    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100 * val_correct / val_total
    val_loss = val_loss / len(val_loader)

    # Print epoch summary
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    # --- Save epoch checkpoint ---
    checkpoint_path = os.path.join(checkpoint_dir, f"epoch_{epoch+1}.pth")
    torch.save({
        'epoch': epoch,
        'batch': len(train_loader)-1,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
    }, checkpoint_path)

    # Update latest checkpoint
    torch.save({
        'epoch': epoch,
        'batch': len(train_loader)-1,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
    }, latest_checkpoint)

    # Save log
    with open(log_file, "a") as f:
        f.write(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
                f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%\n")

# --- Save final model ---
final_model_path = os.path.join(checkpoint_dir, "final_model.pth")
torch.save(model.state_dict(), final_model_path)
print("✅ Final trained model saved at:", final_model_path)
print("✅ Training completed! Checkpoints saved in:", checkpoint_dir)


🔄 Loading latest checkpoint...
✅ Resuming from epoch 1, batch 301


Epoch 1/13:   0%|          | 0/2270 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1/13:   5%|▍         | 103/2270 [07:18<2:29:01,  4.13s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1/13: 100%|██████████| 2270/2270 [1:45:56<00:00,  2.80s/it, Batch Loss=0.6044, Acc=76.44%]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/13] Train Loss: 0.6044, Train Acc: 76.44% | Val Loss: 0.8096, Val Acc: 73.20%


Epoch 2/13:  11%|█         | 244/2270 [07:39<40:05,  1.19s/it, Batch Loss=0.5090, Acc=82.62%]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2/13:  30%|███       | 688/2270 [22:33<49:10,  1.86s/it, Batch Loss=0.4988, Acc=83.09%]  